In [1]:
import json
import pandas as pd
import google.generativeai as genai
import time
import re
import random
import uuid
import os
from dotenv import load_dotenv

C:\Users\nurul\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load file .env yang ada di direktori
load_dotenv()

# Mengambil API key dari environment variables
my_api_key = os.getenv("GEMINI_API_KEY")
new_api_key = os.getenv("NEW_API_KEY")

In [3]:
# # Set up API Key Gemini
# genai.configure(api_key=my_api_key)

# # Kumpulan model flash & lite yang aktif di laptopmu
# MODEL_POOL = [
#     'gemini-2.5-flash-lite',
#     'gemini-2.0-flash',
#     'gemini-flash-latest',
#     'gemini-flash-lite-latest',
#     'gemini-3.1-flash-lite',
#     'gemini-2.0-flash-lite'
# ]

# # Mulai dengan model pertama di dalam list
# current_model_index = 0
# model = genai.GenerativeModel(MODEL_POOL[current_model_index])
# print(f"Mulai menggunakan model utama: {MODEL_POOL[current_model_index]}")

In [4]:
# API Key baru
genai.configure(api_key=new_api_key)

# Kembalikan kumpulan model terbaik kita
MODEL_POOL = [
    'gemini-2.5-flash-lite',
    'gemini-2.0-flash',
    'gemini-flash-latest',
    'gemini-flash-lite-latest',
    'gemini-3.1-flash-lite',
    'gemini-2.0-flash-lite'
]

# PENTING: Reset lagi indexnya ke 0 biar pake kuota segar model utama!
current_model_index = 0
model = genai.GenerativeModel(MODEL_POOL[current_model_index])

print(f"Sukses ganti akun! Mulai ulang dengan model utama: {MODEL_POOL[current_model_index]}")

Sukses ganti akun! Mulai ulang dengan model utama: gemini-2.5-flash-lite


In [5]:
# 1. Load Data
with open('../data/question_seed.json', 'r') as f:
    question_seeds = json.load(f)

with open('../data/scoring_rubric.json', 'r') as f:
    scoring_rubric = json.load(f)

with open('../data/weakness_taxonomy.json', 'r') as f:
    weakness_taxonomy = json.load(f)

# Buat list berisi semua nama kelemahan untuk dimasukkan ke prompt AI
all_weakness_keys = list(weakness_taxonomy.keys())

In [6]:
# Answer 

general_tags = [
    "context_missing", "contribution_unclear", "tools_missing", "impact_missing", 
    "measurable_result_missing", "star_structure_missing", "role_relevance_low", 
    "technical_detail_missing", "communication_unclear", "self_awareness_missing"
]

role_specific_tags_map = {
    "Cloud Engineer": ["security_awareness_missing", "deployment_missing"],
    "DevOps Engineer": ["security_awareness_missing", "deployment_missing", "testing_coverage_missing"],
    "Manual QA Tester": ["testing_coverage_missing"],
    "QA Automation Engineer": ["testing_coverage_missing", "deployment_missing"],
    "Software Tester": ["testing_coverage_missing"],
    "Mobile Developer": ["mobile_platform_detail_missing"],
    "Android Developer": ["mobile_platform_detail_missing"],
    "iOS Developer": ["mobile_platform_detail_missing"],
    "Cybersecurity Analyst": ["security_awareness_missing"]
}

def generate_synthetic_data(role, competency, question, quality_label):
    
    # 1. Siapkan Allowed Tags
    allowed_tags = general_tags + role_specific_tags_map.get(role, [])
    
    # 2. Randomizer Persona & Gaya STT (Filler Words)
    weak_personas = [
        "Sangat gugup, banyak menggunakan filler word (ehh, emm, apa ya), dan meralat kalimatnya sendiri di tengah jalan.",
        "Menjawab terlalu santai pakai bahasa gaul (kayaknya, sih, gitu doang), jawabannya pendek dan terkesan meremehkan/asal bunyi.",
        "Terbata-bata, sering berhenti berpikir panjang, dan akhirnya jawabannya melantur tidak nyambung dengan pertanyaan.",
        "Sering mengulang kata yang sama karena kehabisan ide, dan terkesan menyalahkan situasi pakai bahasa informal."
    ]
    
    avg_personas = [
        "Cukup percaya diri tapi kadang masih ada 'emm' atau 'ehh'. Menjelaskan panjang lebar tapi kurang spesifik pada hasil.",
        "Bicara seperti orang yang sedang mengingat-ingat masa lalu ('waktu itu sih...', 'kalau nggak salah...'), lupa nyebutin metrik.",
        "Bahasanya natural seperti ngobrol biasa ('terus...', 'nah habis itu...'), tapi struktur STAR-nya agak berantakan.",
        "Menjawab lumayan lancar, tapi terlalu sering pakai kata ganti 'kita/kami' (bukan 'saya') dan terdengar kurang menonjolkan diri."
    ]
    
    # 3. Dynamic Prompt Building
    if quality_label == "Weak":
        flaw = random.choice(weak_personas)
        quality_rules = f"""
        Buat jawaban kandidat yang BURUK (Skor komponen 20-49). Evidence level 1 atau 2.
        Skenario gaya bicara: "{flaw}"
        Wajib gunakan filler words (ehh, emm), jeda berpikir, atau kalimat yang tidak tuntas.
        Berdasarkan jawabanmu, pilih semua weakness_tags yang relevan dari list ini: {allowed_tags}.
        """
    elif quality_label == "Average":
        flaw = random.choice(avg_personas)
        quality_rules = f"""
        Buat jawaban kandidat yang CUKUP / BIASA SAJA (Skor komponen 50-74). Evidence level 3 atau 4.
        Skenario gaya bicara: "{flaw}"
        Gunakan gaya bahasa lisan sehari-hari tetapi gaya interview (kemudian, jadi tuh, nah, dll), sisipkan 1-2 filler words natural.
        Berdasarkan jawabanmu, pilih minimal 3 weakness_tags yang paling relevan dari list ini: {allowed_tags}.
        """
    else: # Strong
        quality_rules = f"""
        Buat jawaban kandidat yang SANGAT BAIK (Skor komponen 75-100). Evidence level 4 atau 5.
        Kandidat menjawab dengan percaya diri, terstruktur (STAR method), jelas kontribusinya, dan pakai metrik.
        Gaya bicara tetap lisan natural (bukan seperti baca buku teks), tapi sangat minim filler words dan artikulasinya jelas.
        Kolom weakness_tags HARUS dibiarkan kosong [].
        """

    system_prompt = f"""
    Kamu adalah seorang asisten AI yang bertugas membuat dataset wawancara teknis. Target aplikasinya akan memproses hasil Speech-to-Text (STT).
    Target Role: {role}
    Competency: {competency}
    Pertanyaan: {question}

    TUGAS 1: Buat transkrip STT jawaban kandidat (First person "Saya...").
    {quality_rules}

    TUGAS 2: Lakukan self-annotation terhadap teks jawabanmu sendiri untuk diekstrak menjadi JSON.
    
    OUTPUT WAJIB berupa JSON dengan struktur persis seperti ini, tanpa markdown tambahan:
    {{
        "answer": "Teks transkrip jawaban lisan di sini...",
        "role_relevance": angka_skor_0_sampai_100,
        "star_structure": angka_skor_0_sampai_100,
        "evidence_specificity": angka_skor_0_sampai_100,
        "technical_accuracy": angka_skor_0_sampai_100,
        "communication_clarity": angka_skor_0_sampai_100,
        "self_awareness": angka_skor_0_sampai_100,
        "evidence_level": angka_1_sampai_5,
        "weakness_tags": ["tag1", "tag2"], 
        "has_tool": 1_atau_0,
        "has_metric": 1_atau_0,
        "has_impact": 1_atau_0,
        "has_action": 1_atau_0,
        "has_context": 1_atau_0
    }}
    """
    
    try:
        response = model.generate_content(system_prompt)
        raw_text = response.text.strip()
        
        if raw_text.startswith("```json"):
            raw_text = raw_text[7:-3].strip()
        elif raw_text.startswith("```"):
            raw_text = raw_text[3:-3].strip()
            
        ai_data = json.loads(raw_text)
        return ai_data
    except json.JSONDecodeError as je:
        # Jika hanya gagal parsing JSON karena filler words berantakan, kembalikan None (Aman untuk di-skip)
        print(f"  [!] Teks bukan JSON valid untuk {role} - {quality_label}. Dilewati.")
        return None
    except Exception as e:
        # Jika error dari API Google (seperti 429 Quota Exceeded), lempar errornya keluar!
        raise e

In [7]:
# Scoring Calculation

def calculate_ds_metrics(ai_data, rubric_weights, taxonomy):
    """
    Fungsi ini mengambil raw data dari AI dan melakukan perhitungan matematis
    dan logical rules sesuai dengan standard scoring kamu.
    """
    if not ai_data:
        return None

    # 1. Hitung Final Score sesuai Bobot (Weights)
    weights = rubric_weights['components']
    weighted_score = (
        ai_data.get('role_relevance', 0) * weights['role_relevance']['weight'] +
        ai_data.get('star_structure', 0) * weights['star_structure']['weight'] +
        ai_data.get('evidence_specificity', 0) * weights['evidence_specificity']['weight'] +
        ai_data.get('technical_accuracy', 0) * weights['technical_accuracy']['weight'] +
        ai_data.get('communication_clarity', 0) * weights['communication_clarity']['weight'] +
        ai_data.get('self_awareness', 0) * weights['self_awareness']['weight']
    )
    
    final_score_0_100 = round(weighted_score, 2)
    final_score_0_1 = round(weighted_score / 100, 4)

    # 2. Logika need_clarification (Sesuai need_clarification_rule di rubricmu)
    weak_tags = ai_data.get('weakness_tags', [])
    need_clarif = False
    if final_score_0_1 < 0.75 or ai_data.get('evidence_level', 1) <= 3 or len(weak_tags) > 0:
        need_clarif = True

    # 3. Ambil clarification_type dari weakness_taxonomy
    clarif_type = "none"
    if need_clarif and len(weak_tags) > 0:
        # Ambil tag pertama sebagai prioritas utama untuk diklarifikasi
        first_tag = weak_tags[0]
        if first_tag in taxonomy:
            clarif_type = taxonomy[first_tag]['clarification_type']
    elif need_clarif:
        # Fallback jika butuh klarifikasi tapi AI tidak memberikan tag
        clarif_type = "general"

    # 4. Hitung panjang kata
    answer_text = ai_data.get('answer', '')
    word_count = len(answer_text.split())

    # Kembalikan data yang sudah lengkap
    return {
        "answer": answer_text,
        "role_relevance": ai_data.get('role_relevance'),
        "star_structure": ai_data.get('star_structure'),
        "evidence_specificity": ai_data.get('evidence_specificity'),
        "technical_accuracy": ai_data.get('technical_accuracy'),
        "communication_clarity": ai_data.get('communication_clarity'),
        "self_awareness": ai_data.get('self_awareness'),
        "evidence_level": ai_data.get('evidence_level'),
        "weakness_tags": ";".join(weak_tags), # Ubah list jadi string dipisah titik koma
        "need_clarification": need_clarif,
        "clarification_type": clarif_type,
        "final_score_0_100": final_score_0_100,
        "final_score_0_1": final_score_0_1,
        "has_tool": ai_data.get('has_tool'),
        "has_metric": ai_data.get('has_metric'),
        "has_impact": ai_data.get('has_impact'),
        "has_action": ai_data.get('has_action'),
        "has_context": ai_data.get('has_context'),
        "answer_length_words": word_count
    }

In [ ]:
# Save file 

domain_name = "Information & Technology"
output_file = '../data/answer_quality_dataset_synthetic.csv'

# Urutan kolom sesuai format 
kolom_urut = [
    'sample_id', 'domain', 'role_family', 'target_role', 'competency', 'question', 'answer',
    'role_relevance', 'star_structure', 'evidence_specificity', 'technical_accuracy', 
    'communication_clarity', 'self_awareness', 'evidence_level', 'weakness_tags', 
    'need_clarification', 'clarification_type', 'final_score_0_100', 'final_score_0_1', 
    'quality_label', 'has_tool', 'has_metric', 'has_impact', 'has_action', 'has_context', 
    'answer_length_words'
]

# PENGECEKAN 1 & 2: Cek data yang sudah ada biar bisa Auto-Resume
processed_combinations = set()
if os.path.exists(output_file):
    df_existing = pd.read_csv(output_file)
    # Catat apa saja yang sudah di-generate (Role + Pertanyaan + Kualitas)
    for _, row in df_existing.iterrows():
        combo_id = f"{row['target_role']}_{row['question']}_{row['quality_label']}"
        processed_combinations.add(combo_id)
    print(f"Menemukan {len(processed_combinations)} baris data sebelumnya. Melanjutkan proses...")
else:
    # Bikin file CSV baru dengan header kalau belum ada
    pd.DataFrame(columns=kolom_urut).to_csv(output_file, index=False)
    print("Membuat file CSV baru...")

print("Memulai proses generate data...")

# Iterasi per Role di question_seeds
for role_name, questions in question_seeds.items():
    print(f"\nMemproses Role: {role_name} ({len(questions)} pertanyaan)")
    
    for q_data in questions:
        role_family = q_data.get('role_family')
        competency = q_data.get('competency')
        question_text = q_data.get('question_seed')
        
        for quality in ["Weak", "Average", "Strong"]:
            combo_id = f"{role_name}_{question_text}_{quality}"
            if combo_id in processed_combinations:
                continue 
            
            max_retries = len(MODEL_POOL) # Coba sebanyak model yang tersedia
            for attempt in range(max_retries):
                try:
                    # 1. AI Generate Data
                    ai_output = generate_synthetic_data(
                        role=role_name, 
                        competency=competency, 
                        question=question_text, 
                        quality_label=quality
                    )
                    
                    if ai_output:
                        calculated_data = calculate_ds_metrics(ai_output, scoring_rubric, weakness_taxonomy)
                        
                        if calculated_data:
                            unique_id = f"AQD-{str(uuid.uuid4().hex)[:6].upper()}"
                            
                            row = {
                                "sample_id": unique_id,
                                "domain": domain_name,
                                "role_family": role_family,
                                "target_role": role_name,
                                "competency": competency,
                                "question": question_text,
                                **calculated_data,
                                "quality_label": quality
                            }
                            
                            df_row = pd.DataFrame([row])
                            df_row = df_row[kolom_urut]
                            df_row.to_csv(output_file, mode='a', header=False, index=False)
                            print(f"  [+] Saved: {role_name} | {competency[:15]}... | {quality} (via {MODEL_POOL[current_model_index]})")
                    
                    time.sleep(4) # Jeda standar antar request sukses
                    break 
                    
                except Exception as e:
                    # JIKA KENA LIMIT HARIAN (429), JANGAN TIDUR! LANGSUNG GANTI MODEL!
                    if "429" in str(e) or "quota" in str(e).lower():
                        current_model_index += 1
                        
                        # Jika semua model di pool sudah habis kena limit
                        if current_model_index >= len(MODEL_POOL):
                            print("\n[🚨] Semua model cadangan sudah terkena limit harian Google! Proses terpaksa berhenti.")
                            raise e
                        
                        print(f"\n[🔄] Model {MODEL_POOL[current_model_index-1]} kena limit harian. Otomatis switch ke: {MODEL_POOL[current_model_index]}")
                        # Ganti objek model global ke model cadangan berikutnya
                        model = genai.GenerativeModel(MODEL_POOL[current_model_index])
                        continue # Langsung coba ulang pertanyaan yang sama pakai model baru
                    else:
                        # Jika errornya bukan 429 (misal internet putus), baru tidur sebentar
                        print(f"  [!] Error non-kuota: {e}. Menunggu 10 detik...")
                        time.sleep(10)
                    
print(f"\nProses selesai sepenuhnya!")

Menemukan 545 baris data sebelumnya. Melanjutkan proses...
Memulai proses generate data...

Memproses Role: Data Analyst (21 pertanyaan)

Memproses Role: Data Scientist (21 pertanyaan)

Memproses Role: AI Engineer (21 pertanyaan)

Memproses Role: ML Engineer (21 pertanyaan)

Memproses Role: Cloud Engineer (21 pertanyaan)

Memproses Role: DevOps Engineer (21 pertanyaan)

Memproses Role: Cybersecurity Analyst (21 pertanyaan)

Memproses Role: Frontend Developer (21 pertanyaan)

Memproses Role: Backend Developer (21 pertanyaan)
  [+] Saved: Backend Developer | Microservices A... | Strong (via gemini-2.5-flash-lite)
  [!] Teks bukan JSON valid untuk Backend Developer - Weak. Dilewati.
  [!] Teks bukan JSON valid untuk Backend Developer - Average. Dilewati.
  [!] Teks bukan JSON valid untuk Backend Developer - Strong. Dilewati.
  [+] Saved: Backend Developer | Database Tuning... | Weak (via gemini-2.5-flash-lite)
  [!] Teks bukan JSON valid untuk Backend Developer - Average. Dilewati.
  [!] 

In [9]:
print("Daftar nama model yang BISA kamu pakai saat ini:")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        # Kita hapus 'models/' karena genai.GenerativeModel otomatis menambahkannya
        clean_name = m.name.replace('models/', '')
        print(f"- '{clean_name}'")

Daftar nama model yang BISA kamu pakai saat ini:
- 'gemini-2.5-flash'
- 'gemini-2.5-pro'
- 'gemini-2.0-flash'
- 'gemini-2.0-flash-001'
- 'gemini-2.0-flash-lite-001'
- 'gemini-2.0-flash-lite'
- 'gemini-2.5-flash-preview-tts'
- 'gemini-2.5-pro-preview-tts'
- 'gemma-4-26b-a4b-it'
- 'gemma-4-31b-it'
- 'gemini-flash-latest'
- 'gemini-flash-lite-latest'
- 'gemini-pro-latest'
- 'gemini-2.5-flash-lite'
- 'gemini-2.5-flash-image'
- 'gemini-3-pro-preview'
- 'gemini-3-flash-preview'
- 'gemini-3.1-pro-preview'
- 'gemini-3.1-pro-preview-customtools'
- 'gemini-3.1-flash-lite-preview'
- 'gemini-3.1-flash-lite'
- 'gemini-3-pro-image-preview'
- 'nano-banana-pro-preview'
- 'gemini-3.1-flash-image-preview'
- 'gemini-3.5-flash'
- 'lyria-3-clip-preview'
- 'lyria-3-pro-preview'
- 'gemini-3.1-flash-tts-preview'
- 'gemini-robotics-er-1.5-preview'
- 'gemini-robotics-er-1.6-preview'
- 'gemini-2.5-computer-use-preview-10-2025'
- 'antigravity-preview-05-2026'
- 'deep-research-max-preview-04-2026'
- 'deep-research